# 🎬 Workflow End-to-End — Demo en vivo del pipeline completo

Este notebook orquesta la **demostración en vivo** exigida por la sección 5
de la guía del proyecto:

> *"Demostración en vivo del flujo completo Bronze → Silver → Gold. El dato
> debe recorrer todas las capas de forma observable."*

## Arquitectura del flujo

```
┌─────────────────────────────────────────────────────────────┐
│  PASO 1 — INGESTA (notebook 06)                              │
│  Simulación de 155 eventos sintéticos en las 6 tablas Bronze │
│  • 5 destinos + 30 usuarios + 10 propiedades                 │
│  • 50 reservas + 40 pagos + 20 reseñas                       │
└─────────────────────────┬───────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────────────┐
│  PASO 2 — TRANSFORMACIÓN (notebook 08)                       │
│  dbt limpia y construye Silver + Gold con los nuevos eventos │
│  • dbt run --select silver  → 6 tablas Silver actualizadas   │
│  • dbt run --select gold    → Star Schema actualizado        │
│  • dbt test                 → 30 tests de calidad            │
└─────────────────────────┬───────────────────────────────────┘
                          │
                          ▼
┌─────────────────────────────────────────────────────────────┐
│  PASO 3 — VISUALIZACIÓN (Power BI)                           │
│  Refrescar dashboards → KPIs actualizados (GMV, Revenue, etc)│
│  Conexión vía DirectQuery al SQL Warehouse de Databricks     │
└─────────────────────────────────────────────────────────────┘
```

## Cómo usar este notebook

**Para la sustentación en vivo:**
1. Abre este notebook en Databricks.
2. Clic en **Run all**.
3. Espera ~3-5 minutos a que termine.
4. Mostrar el resumen al final.
5. Cambiar a Power BI → presionar Refresh.
6. Los KPIs de los dashboards reflejan los nuevos eventos.

## Ventajas vs ejecutar los notebooks por separado

- **Demo coherente**: un solo Run all dispara todo el pipeline.
- **Observable**: cada paso imprime su progreso y tiempo de ejecución.
- **Auditoría**: el resumen final muestra conteos antes/después.
- **Programable**: este notebook se puede configurar como Databricks Job
  con schedule (por ejemplo, ejecutarlo cada hora) — ver `documentation/databricks_workflow.md`.

## 0. Estado INICIAL — conteo antes del pipeline

Capturamos cuántos registros hay AHORA en cada capa, para luego comparar.

In [ ]:
import time
from datetime import datetime

workflow_start = time.time()
print(f"🎬 INICIANDO WORKFLOW END-TO-END a las {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

# Conteos iniciales
conteos_antes = {}
for tabla in ["bronze.bronze_destinations", "bronze.bronze_users",
              "bronze.bronze_properties", "bronze.bronze_bookings",
              "bronze.bronze_payments", "bronze.bronze_reviews",
              "silver.silver_bookings", "gold.gold_fact_reservas"]:
    try:
        count = spark.table(tabla).count()
        conteos_antes[tabla] = count
        print(f"  {tabla:<40} {count:>10,} registros")
    except Exception as e:
        conteos_antes[tabla] = 0
        print(f"  {tabla:<40} {'N/A':>10}  (tabla no existe aún)")

print("=" * 70)

## PASO 1/3 — Simulación de eventos en Bronze

Ejecuta el notebook `06_streaming_pipeline` que inserta eventos sintéticos
en las 6 tablas Bronze (destinations, users, properties, bookings, payments, reviews).

In [ ]:
paso_start = time.time()
print("━" * 70)
print("PASO 1/3 — Ejecutando notebook 06 (Streaming Pipeline)")
print("━" * 70)

result_06 = dbutils.notebook.run(
    "./06_streaming_pipeline",
    timeout_seconds=900
)

print(f"\n✅ PASO 1 completado en {time.time() - paso_start:.1f} segundos")
print(f"   Resultado: {result_06}")

## PASO 2/3 — Limpieza y transformación con dbt

Ejecuta el notebook `08_run_dbt` que corre:
- `dbt run --select silver` (procesa las 6 tablas con los nuevos eventos)
- `dbt run --select gold` (reconstruye el Star Schema)
- `dbt test` (valida calidad de datos)
- `dbt docs generate` (actualiza el linaje)

In [ ]:
paso_start = time.time()
print("━" * 70)
print("PASO 2/3 — Ejecutando notebook 08 (dbt run + test + docs)")
print("━" * 70)

result_08 = dbutils.notebook.run(
    "./08_run_dbt",
    timeout_seconds=900
)

print(f"\n✅ PASO 2 completado en {time.time() - paso_start:.1f} segundos")
print(f"   Resultado: {result_08}")

## 1. Estado FINAL — conteo después del pipeline

Comparamos los conteos antes y después para validar que el flujo funcionó.

In [ ]:
print("=" * 70)
print(f"📊 RESUMEN — Comparación ANTES vs DESPUÉS")
print("=" * 70)
print(f"{'Tabla':<40} {'Antes':>10} {'Después':>10} {'Δ':>10}")
print("─" * 70)

for tabla in ["bronze.bronze_destinations", "bronze.bronze_users",
              "bronze.bronze_properties", "bronze.bronze_bookings",
              "bronze.bronze_payments", "bronze.bronze_reviews",
              "silver.silver_bookings", "gold.gold_fact_reservas"]:
    try:
        count_despues = spark.table(tabla).count()
        count_antes = conteos_antes.get(tabla, 0)
        delta = count_despues - count_antes
        delta_str = f"+{delta:,}" if delta > 0 else f"{delta:,}"
        print(f"  {tabla:<40} {count_antes:>10,} {count_despues:>10,} {delta_str:>10}")
    except Exception as e:
        print(f"  {tabla:<40} {'ERROR':>10}")

print("=" * 70)
print(f"⏱️  TIEMPO TOTAL DEL WORKFLOW: {time.time() - workflow_start:.1f} segundos")
print("=" * 70)

## PASO 3/3 — Refrescar Power BI (manual)

**Acción del usuario**: cambiar a Power BI Desktop y presionar **Actualizar**.

Los dashboards conectados vía DirectQuery al SQL Warehouse de Databricks
van a leer las tablas Gold actualizadas (`gold.gold_fact_reservas` y dimensiones)
y refrescar los KPIs en tiempo real:

- **GMV Bruto**: aumenta proporcional al valor de las 50 nuevas reservas.
- **Revenue Neto**: aumenta con las reservas confirmadas nuevas.
- **Reservas Totales**: +50 reservas.
- **Usuarios Totales**: +30 usuarios.
- **Propiedades Activas**: +10 propiedades.

## Conclusión

### Lo que demuestra este workflow

Este notebook orquesta la **demostración en vivo end-to-end** exigida por la
guía del proyecto:

1. ✅ **Ingesta** — eventos nuevos llegan a Bronze (notebook 06).
2. ✅ **Transformación** — dbt limpia y construye Silver + Gold (notebook 08).
3. ✅ **Visualización** — Power BI refleja los nuevos datos en vivo.

### Cómo programarlo como Databricks Job (sustentación pro)

Para una demo aún más profesional, este notebook se puede configurar como
**Databricks Job** con schedule automático:

1. Menú izquierdo → **Jobs & Pipelines** → **Create job**.
2. Task type: **Notebook**.
3. Notebook path: `/Workspace/Users/.../09_workflow_demo`.
4. Schedule: cada hora, cada día, o manual.
5. Notifications: email cuando se complete.

Ver guía detallada en `documentation/databricks_workflow.md` (incluido en el repo).

### Para la sustentación

> *"Implementamos un workflow orquestador (notebook 09) que demuestra el
> pipeline end-to-end en una sola ejecución: inserta 155 eventos sintéticos
> en Bronze, los procesa con dbt para construir Silver y Gold limpios, ejecuta
> 30 tests de calidad, y refresca el linaje. Todo el flujo es observable y
> auditable — el notebook compara conteos antes y después para validar que
> los datos recorrieron todas las capas correctamente. Programable como
> Databricks Job con schedule automático."*

Eso es lo que pide la sección 5 de la guía. Demo completa.